In [ ]:
from pathlib import Path
import os
import sys

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / 'environment/layout.json').is_file())
sys.path.insert(0, str(PROJECT / 'code'))
from notebook_support import prepare_runtime, install, execute_module_cell, run_figure_module
RUNTIME = prepare_runtime(PROJECT)
sys.path.insert(0, str(RUNTIME))
os.chdir(RUNTIME)
install()
get_ipython().register_magic_function(execute_module_cell, 'cell', 'plot_module')


## Figure style


In [ ]:
%%plot_module pvsim.viz
"""Shared figure style."""

from __future__ import annotations

from pvsim.labels import label as _text_label

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm


COLORS = {
    "c-Si": "#1f6fb2",
    "c-Si-modern": "#1f6fb2",
    "perovskite": "#e2641e",
    "tandem": "#2a9d4a",
}
LABEL_CN = {"c-Si": _text_label('tech_early_csi'), "c-Si-modern": _text_label('tech_modern_csi'),
            "perovskite": _text_label('tech_perovskite'), "tandem": _text_label('viz_text')}


In [ ]:
%%plot_module pvsim.viz
def setup(font_size: int = 12):

    avail = set(f.name for f in fm.fontManager.ttflist)
    for cand in ["Microsoft YaHei", "SimHei", "DengXian", "SimSun"]:
        if cand in avail:
            plt.rcParams["font.sans-serif"] = [cand]
            break
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["font.size"] = font_size
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
    plt.rcParams["figure.dpi"] = 110
    plt.rcParams["savefig.bbox"] = "tight"


In [ ]:
%%plot_module pvsim.viz
from pvsim.labels import label as _text_label
def setup_en(font_size: int = 12):

    avail = set(f.name for f in fm.fontManager.ttflist)
    for cand in ["Arial", "Helvetica", "DejaVu Sans", "Liberation Sans"]:
        if cand in avail:
            plt.rcParams["font.sans-serif"] = [cand]
            break
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["font.size"] = font_size
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
    plt.rcParams["figure.dpi"] = 110
    plt.rcParams["savefig.bbox"] = "tight"


LABEL_EN = {"c-Si": "c-Si", "perovskite": "Perovskite", "tandem": "Tandem"}
TECH_EN = {_text_label('tech_csi'): "c-Si", _text_label('tech_perovskite'): "Perovskite", _text_label('tech_tandem'): "Tandem"}


In [ ]:
%%plot_module pvsim.viz
def color(tech_name: str) -> str:
    return COLORS.get(tech_name, "#888888")


In [ ]:
%%plot_module pvsim.viz
def save(fig, path: str):
    fig.savefig(path, dpi=130)
    plt.close(fig)
    return path


## Energy yield and technology comparison


In [ ]:
%%plot_module scripts.portfolio_physics
"""Energy yields, lifetime costs and technology comparisons."""

from pvsim.labels import label as _text_label

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pvsim import viz
from pvsim.economic_priors import (
    CENTRAL_BREAKTHROUGH_YEAR,
    CENTRAL_CAPEX_FLOORS_USD_W,
    CENTRAL_LEARNING_RATES,
    CENTRAL_POST_BREAKTHROUGH_LIFETIME_YR,
    INITIAL_GLOBAL_DEPLOYMENT_GW,
    INITIAL_SYSTEM_CAPEX_USD_W,
    TECH_CSI,
    TECH_PEROVSKITE,
    TECH_TANDEM,
)
from pvsim.economics import (
    DISCOUNT_RATE,
    MODERN_CSI_DURABILITY,
    OPEX_FRACTION_OF_CAPEX,
    TANDEM_DURABILITY,
    discounted_lcoe,
    perovskite_durability,
)
from pvsim.materials import CSI_MODERN, PEROVSKITE, TANDEM_2T
from pvsim.cities import CITIES
from pvsim import weather as wx
from pvsim.system import SystemConfig, simulate

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

TECHS = [("c-Si", _text_label('tech_csi'), CSI_MODERN), ("perovskite", _text_label('tech_perovskite'), PEROVSKITE),
         ("tandem", _text_label('tech_tandem'), TANDEM_2T)]
COLORS = {_text_label('tech_csi'): "#1f6fb2", _text_label('tech_perovskite'): "#e2641e", _text_label('tech_tandem'): "#2a9d4a"}
YEARS = np.arange(2025, 2051)
N = len(YEARS)


LR = dict(CENTRAL_LEARNING_RATES)
B = {k: -np.log2(1 - v) for k, v in LR.items()}
CAPEX_0 = dict(INITIAL_SYSTEM_CAPEX_USD_W)
Q_0 = dict(INITIAL_GLOBAL_DEPLOYMENT_GW)
CAPEX_FLOOR = dict(CENTRAL_CAPEX_FLOORS_USD_W)
MAX_DROP = 0.12
DISCOUNT = DISCOUNT_RATE
OPEX = OPEX_FRACTION_OF_CAPEX


DEG_0 = {_text_label('tech_csi'): MODERN_CSI_DURABILITY.degradation_rate,
         _text_label('tech_perovskite'): PEROVSKITE.degradation_rate,
         _text_label('tech_tandem'): TANDEM_DURABILITY.degradation_rate}
BURN_0 = {_text_label('tech_csi'): MODERN_CSI_DURABILITY.burn_in_loss,
          _text_label('tech_perovskite'): PEROVSKITE.burn_in_loss,
          _text_label('tech_tandem'): TANDEM_DURABILITY.burn_in_loss}
LIFE_0 = {_text_label('tech_csi'): 25.0, _text_label('tech_perovskite'): 15.0, _text_label('tech_tandem'): 25.0}
LIFE_FINAL_PEROV = CENTRAL_POST_BREAKTHROUGH_LIFETIME_YR
BREAKTHROUGH_YEAR = CENTRAL_BREAKTHROUGH_YEAR
_FINAL_PEROV = perovskite_durability(
    BREAKTHROUGH_YEAR,
    BREAKTHROUGH_YEAR,
    LIFE_FINAL_PEROV,
)
DEG_FINAL_PEROV = _FINAL_PEROV.degradation_rate
BURN_FINAL_PEROV = _FINAL_PEROV.burn_in_loss


In [ ]:
%%plot_module scripts.portfolio_physics
def perov_evolve(year, breakthrough_year=BREAKTHROUGH_YEAR):

    state = perovskite_durability(
        year,
        breakthrough_year,
        LIFE_FINAL_PEROV,
    )
    return {
        "life": state.lifetime_years,
        "deg": state.degradation_rate,
        "burn": state.burn_in_loss,
    }


In [ ]:
%%plot_module scripts.portfolio_physics
from pvsim.labels import label as _text_label
def tech_year_params(tech, year):

    if tech == _text_label('tech_perovskite'):
        e = perov_evolve(year)
        return e["life"], e["deg"], e["burn"]
    return LIFE_0[tech], DEG_0[tech], BURN_0[tech]


In [ ]:
%%plot_module scripts.portfolio_physics
def yield_physics_year0():

    print('[1/4] Physical simulation: 12 cities × 3 technologies × 8760h...')
    cfg = SystemConfig(n_modules=20)
    results = []
    for city in CITIES:
        w = wx.from_pvgis_tmy(city.lat, city.lon, altitude=city.alt, name=city.key)
        tair_mean = float(w["temp_air"].mean())
        ghi_total = float(w["ghi"].sum() / 1000.0)    # kWh/m²
        for code, name, tech in TECHS:
            r = simulate(tech, w, cfg, npts=60)
            ts = r["timeseries"]
            poa = ts["poa_global"].to_numpy(); tcell = ts["tcell"].to_numpy()
            sf = ts["spectral_factor"].to_numpy()
            m = poa > 50
            tcell_w = float(np.average(tcell[m], weights=poa[m])) if m.any() else np.nan
            sf_w = float(np.average(sf[m], weights=poa[m])) if m.any() else np.nan

            area_m2 = 20 * tech.cells_in_series * tech.area_cm2 * 1e-4  # 20 modules
            kwh_per_m2 = r["energy_ac_kwh"] / area_m2
            results.append({
                "city": city.name, "key": city.key, "lat": city.lat, "alt": city.alt,
                "tech": name, "code": code,
                "kwp": r["kwp"],
                "yield_kwh_per_kwp": r["specific_yield"],
                "yield_kwh_per_m2": kwh_per_m2,
                "PR": r["performance_ratio"],
                "tcell_weighted": tcell_w,
                "spectral_factor_w": sf_w,
                "ghi_kwh_m2": ghi_total,
                "tair_mean": tair_mean,
            })
        print(f"  {city.name:5s}: c-Si {results[-3]['yield_kwh_per_kwp']:.0f} | "
              f"perov {results[-2]['yield_kwh_per_kwp']:.0f} | "
              f"tandem {results[-1]['yield_kwh_per_kwp']:.0f} kWh/kWp")
    return pd.DataFrame(results)


In [ ]:
%%plot_module scripts.portfolio_physics
def capex_path_global(deployment):

    cum = {k: Q_0[k] for k in CAPEX_0}
    cap_eff = {k: CAPEX_0[k] for k in CAPEX_0}
    path = {k: np.zeros(N) for k in CAPEX_0}
    for i in range(N):
        for k in CAPEX_0:
            raw = max(CAPEX_FLOOR[k],
                      CAPEX_0[k] * (max(cum[k], 0.1) / Q_0[k]) ** (-B[k]))
            if i == 0:
                cap_eff[k] = raw
            else:
                min_allowed = cap_eff[k] * (1 - MAX_DROP)
                cap_eff[k] = max(min_allowed, raw, CAPEX_FLOOR[k])
            path[k][i] = cap_eff[k]
            cum[k] += deployment.get(k, np.zeros(N))[i] * 1.4   # +ROW factor
    return path


In [ ]:
%%plot_module scripts.portfolio_physics
def lcoe_npv(capex_per_w, yield_kwh_per_kwp, life, deg_rate, burn_in):

    return discounted_lcoe(
        capex_per_w,
        yield_kwh_per_kwp,
        life,
        deg_rate,
        burn_in,
    )


In [ ]:
%%plot_module scripts.portfolio_physics
from pvsim.labels import label as _text_label
def build_city_year_lcoe(yield_df):

    print('\n[2/4] LCOE Matrix (12 cities × 3 technologies × 26 year)...')

    base_dep = {_text_label('tech_csi'): np.linspace(80, 30, N) + np.linspace(0, 20, N),
                _text_label('tech_perovskite'): np.minimum(np.arange(N) * 4 + 5, 100),
                _text_label('tech_tandem'): np.maximum(0, np.minimum(np.arange(N) * 3 - 12, 80))}
    cap_path = capex_path_global(base_dep)

    rows = []
    for _, row in yield_df.iterrows():
        for i, yr in enumerate(YEARS):
            cap = cap_path[row["tech"]][i]
            life, deg, burn = tech_year_params(row["tech"], int(yr))
            lc = lcoe_npv(cap, row["yield_kwh_per_kwp"], life, deg, burn)

            cap_per_m2 = cap * 1000 * (row["kwp"] / (20 * row["tech_area_m2"] if "tech_area_m2" in row else 1))
            rows.append({
                "city": row["city"], "key": row["key"], "tech": row["tech"],
                "year": int(yr), "capex_usd_per_w": cap,
                "lcoe_utility_cents_per_kwh": lc * 100,
                "yield_kwh_per_kwp_year0": row["yield_kwh_per_kwp"],
                "yield_kwh_per_m2_year0": row["yield_kwh_per_m2"],
                "tcell_weighted": row["tcell_weighted"],
            })
    df = pd.DataFrame(rows)
    print(f'  Generate {len(df)} rows (12×3×26)')
    return df, cap_path


In [ ]:
%%plot_module scripts.portfolio_physics
from pvsim.labels import label as _text_label
def find_substitution_year(lcoe_df):

    print('\n[3/4] Determine crossover years by city...')
    out = []
    for city in lcoe_df["city"].unique():
        d = lcoe_df[lcoe_df["city"] == city].pivot(index="year", columns="tech",
                                                    values="lcoe_utility_cents_per_kwh")
        crosses = {}
        for src, dst in [(_text_label('tech_csi'), _text_label('tech_perovskite')), (_text_label('tech_csi'), _text_label('tech_tandem')), (_text_label('tech_perovskite'), _text_label('tech_tandem'))]:
            mask = d[dst] < d[src]
            if mask.any():
                crosses[f"{dst}{_text_label('beats')}{src}"] = int(d.index[mask][0])
            else:
                crosses[f"{dst}{_text_label('beats')}{src}"] = None
        row = {"city": city}; row.update(crosses)
        out.append(row)
    return pd.DataFrame(out)


In [ ]:
%%plot_module scripts.portfolio_physics
from pvsim.labels import label as _text_label
def plot_fig37_temperature_advantage(yield_df):

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
    pivot_y = yield_df.pivot(index="city", columns="tech", values="yield_kwh_per_kwp")
    pivot_t = yield_df.pivot(index="city", columns="tech", values="tcell_weighted")
    pivot_y = pivot_y.reindex(yield_df.groupby("city")["tcell_weighted"].first().sort_values().index)
    pivot_t = pivot_t.loc[pivot_y.index]


    ax = axes[0]
    tc = pivot_t[_text_label('tech_csi')].values
    perov_adv = (pivot_y[_text_label('tech_perovskite')] / pivot_y[_text_label('tech_csi')] - 1) * 100
    tand_adv = (pivot_y[_text_label('tech_tandem')] / pivot_y[_text_label('tech_csi')] - 1) * 100
    ax.scatter(tc, perov_adv, s=110, color=COLORS[_text_label('tech_perovskite')], label='Perovskite vs c-Si', zorder=3)
    ax.scatter(tc, tand_adv, s=110, color=COLORS[_text_label('tech_tandem')], label='Tandem vs c-Si', zorder=3, marker="s")
    for city, x, y in zip(pivot_y.index, tc, perov_adv):
        ax.annotate(city, (x, y), textcoords="offset points", xytext=(6, 4), fontsize=8)

    for advs, col in [(perov_adv.values, COLORS[_text_label('tech_perovskite')]),
                       (tand_adv.values, COLORS[_text_label('tech_tandem')])]:
        z = np.polyfit(tc, advs, 1)
        xx = np.linspace(tc.min(), tc.max(), 50)
        ax.plot(xx, np.polyval(z, xx), "--", color=col, alpha=0.6,
                label=f'  Slope {z[0]:+.3f}%/°C')
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_xlabel('Irradiance-weighted cell temperature (°C)')
    ax.set_ylabel('yield Advantage vs c-Si (%)')
    ax.set_title('(a) Effect of temperature-coefficient differences on yield Advantage (Physical 8760h Simulated)', fontweight="bold")
    ax.grid(alpha=0.3); ax.legend(loc="upper left", fontsize=9)


    ax = axes[1]
    pivot_m2 = yield_df.pivot(index="city", columns="tech",
                               values="yield_kwh_per_m2").loc[pivot_y.index]
    x = np.arange(len(pivot_m2)); bw = 0.27
    ax.bar(x - bw, pivot_m2[_text_label('tech_csi')], width=bw, color=COLORS[_text_label('tech_csi')], label='c-Si (η=15.3%)')
    ax.bar(x, pivot_m2[_text_label('tech_perovskite')], width=bw, color=COLORS[_text_label('tech_perovskite')], label='Perovskite (η=19.3%)')
    ax.bar(x + bw, pivot_m2[_text_label('tech_tandem')], width=bw, color=COLORS[_text_label('tech_tandem')], label='Tandem (η=28.3%)')
    ax.set_xticks(x); ax.set_xticklabels(pivot_m2.index, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel('Annual generation (kWh/m²)')
    ax.set_title('(b) Area-normalized: rooftop Tandem advantage under this constraint (1.87× c-Si)',
                 fontweight="bold")
    ax.legend(loc="upper left", fontsize=9); ax.grid(alpha=0.3, axis="y")

    fig.suptitle('Physical model results: utility and rooftop Different technology-choice criteria',
                 fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig("outputs/figures/37_physics_temperature_advantage.png", dpi=130,
                bbox_inches="tight")
    plt.close(fig)


In [ ]:
%%plot_module scripts.portfolio_physics
from pvsim.labels import label as _text_label
def plot_fig38_spectral_mismatch(yield_df):

    fig, ax = plt.subplots(figsize=(13, 5.5))
    pivot = yield_df.pivot(index="city", columns="tech", values="spectral_factor_w")
    pivot_t = yield_df.pivot(index="city", columns="tech", values="tcell_weighted")
    order = pivot_t[_text_label('tech_csi')].sort_values().index
    pivot = pivot.loc[order]
    x = np.arange(len(pivot)); bw = 0.27
    ax.bar(x - bw, pivot[_text_label('tech_csi')], width=bw, color=COLORS[_text_label('tech_csi')], label='c-Si (350-1110nm)')
    ax.bar(x, pivot[_text_label('tech_perovskite')], width=bw, color=COLORS[_text_label('tech_perovskite')], label='Perovskite (350-800nm, narrow)')
    ax.bar(x + bw, pivot[_text_label('tech_tandem')], width=bw, color=COLORS[_text_label('tech_tandem')], label='Tandem (350-1110nm, series)')
    ax.axhline(1.0, color="gray", ls=":", lw=1, alpha=0.7, label='STC Reference (AM1.5G)')
    ax.set_xticks(x); ax.set_xticklabels(pivot.index, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel('Irradiance-weighted spectral factor (Actual spectrum / AM1.5G)')
    ax.set_title('Spectral mismatch map: Perovskite response under blue-rich spectra, Tandem series-current constraint',
                 fontweight="bold")
    ax.set_ylim(pivot.values.min() * 0.99, pivot.values.max() * 1.01)
    ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig("outputs/figures/38_physics_spectral.png", dpi=130, bbox_inches="tight")
    plt.close(fig)


In [ ]:
%%plot_module scripts.portfolio_physics
from pvsim.labels import label as _text_label
def plot_fig39_city_lcoe_evolution(lcoe_df):

    sel = [_text_label('lhasa'), _text_label('dunhuang'), _text_label('beijing'), _text_label('shanghai'), _text_label('guangzhou'), _text_label('haikou')]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), sharey=True)
    axes = axes.flatten()
    for ax, city in zip(axes, sel):
        d = lcoe_df[lcoe_df["city"] == city]
        for tech in [_text_label('tech_csi'), _text_label('tech_perovskite'), _text_label('tech_tandem')]:
            sub = d[d["tech"] == tech].sort_values("year")
            ax.plot(sub["year"], sub["lcoe_utility_cents_per_kwh"],
                    lw=2.2, color=COLORS[tech], label=tech)
        ax.set_title(city, fontweight="bold")
        ax.grid(alpha=0.3); ax.set_xlim(2025, 2050)
        ax.set_xlabel('year'); ax.set_ylabel('LCOE cents/kWh')
    axes[0].legend(loc="upper right", fontsize=9)
    fig.suptitle('City-level LCOE Evolution (Physical model yield + Wright capex + NPV Discounted)',
                 fontweight="bold", y=1.00)
    fig.tight_layout()
    fig.savefig("outputs/figures/39_physics_city_lcoe.png", dpi=130, bbox_inches="tight")
    plt.close(fig)


In [ ]:
%%plot_module scripts.portfolio_physics
from pvsim.labels import label as _text_label
def plot_fig40_substitution_map(cross_df, yield_df, lcoe_df):

    coords = {c.name: (c.lon, c.lat) for c in CITIES}


    d2050 = lcoe_df[lcoe_df["year"] == 2050].pivot(
        index="city", columns="tech", values="lcoe_utility_cents_per_kwh")
    d2050["best_lcoe"] = d2050.min(axis=1)
    d2050["best_tech"] = d2050[[_text_label('tech_csi'), _text_label('tech_perovskite'), _text_label('tech_tandem')]].idxmin(axis=1)
    d2050 = d2050.reset_index()
    d2050["lon"] = d2050["city"].map(lambda c: coords[c][0])
    d2050["lat"] = d2050["city"].map(lambda c: coords[c][1])

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))


    ax = axes[0]
    sc = ax.scatter(d2050["lon"], d2050["lat"], c=d2050["best_lcoe"],
                    s=300, cmap="viridis_r", vmin=1.8, vmax=4.0,
                    edgecolors="black", linewidth=0.9, zorder=3)
    for _, r in d2050.iterrows():
        ax.annotate(f"{r['city']}\n{r['best_lcoe']:.2f}¢",
                    (r["lon"], r["lat"]), textcoords="offset points",
                    xytext=(8, 5), fontsize=9, fontweight="bold")
    ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
    ax.set_title('(a) 2050 Optimal LCOE Geographical distribution (cents/kWh)', fontweight="bold")
    ax.set_xlim(80, 135); ax.set_ylim(18, 50); ax.grid(alpha=0.3)
    plt.colorbar(sc, ax=ax, label='LCOE (cents/kWh)')


    ax = axes[1]
    tech_color = {_text_label('tech_csi'): COLORS[_text_label('tech_csi')], _text_label('tech_perovskite'): COLORS[_text_label('tech_perovskite')], _text_label('tech_tandem'): COLORS[_text_label('tech_tandem')]}
    for tech in [_text_label('tech_csi'), _text_label('tech_perovskite'), _text_label('tech_tandem')]:
        sub = d2050[d2050["best_tech"] == tech]
        if len(sub) > 0:
            ax.scatter(sub["lon"], sub["lat"], c=tech_color[tech], s=350,
                       edgecolors="black", linewidth=0.9, zorder=3,
                       label=f'{tech} Optimal ({len(sub)} cities)')
    for _, r in d2050.iterrows():
        margin = (r[_text_label('tech_csi')] - r["best_lcoe"]) / r["best_lcoe"] * 100
        ax.annotate(f"{r['city']}\n{r['best_tech']}\n(-{margin:.0f}{_text_label('portfolio_physics_text')}",
                    (r["lon"], r["lat"]), textcoords="offset points",
                    xytext=(8, 5), fontsize=8, fontweight="bold")
    ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
    ax.set_title('(b) 2050 Lowest-cost technology by city (utility-scale)', fontweight="bold")
    ax.set_xlim(80, 135); ax.set_ylim(18, 50); ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=9)
    fig.suptitle('2050 Final state: Physical LCOE Geographical pattern + Optimal technology distribution',
                 fontweight="bold", y=1.01)
    fig.tight_layout()
    fig.savefig("outputs/figures/40_physics_2050_landscape.png", dpi=130,
                bbox_inches="tight")
    plt.close(fig)


In [ ]:
%%plot_module scripts.portfolio_physics
from pvsim.labels import label as _text_label
def plot_fig41_rooftop_vs_utility(yield_df, lcoe_df):

    # rooftop LCOE: capex × area / yield_per_m²


    cfg = SystemConfig(n_modules=20)

    sel_cities = [_text_label('lhasa'), _text_label('haikou')]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    for ax, city in zip(axes, sel_cities):

        for tech_name in [_text_label('tech_csi'), _text_label('tech_perovskite'), _text_label('tech_tandem')]:
            tech_data = yield_df[(yield_df["city"] == city) &
                                 (yield_df["tech"] == tech_name)].iloc[0]
            lc_data = lcoe_df[(lcoe_df["city"] == city) &
                              (lcoe_df["tech"] == tech_name)].sort_values("year")
            cap_path = lc_data["capex_usd_per_w"].values

            util_lcoe = lc_data["lcoe_utility_cents_per_kwh"].values

            #   cap_per_m² = cap_per_W × η_STC × 1000 (W/m²)

            eta_rel = {_text_label('tech_csi'): 0.153, _text_label('tech_perovskite'): 0.193, _text_label('tech_tandem'): 0.283}[tech_name]
            cap_per_m2 = cap_path * eta_rel * 1000   # $/m²
            life_t, deg_t, _ = tech_year_params(tech_name, 2040)
            crf = DISCOUNT * (1 + DISCOUNT) ** life_t / \
                  ((1 + DISCOUNT) ** life_t - 1)
            yield_m2 = tech_data["yield_kwh_per_m2"] * \
                       (1 - life_t * deg_t / 2)
            roof_lcoe = cap_per_m2 * (crf + OPEX) / yield_m2 * 100  # cents/kWh
            ax.plot(lc_data["year"], util_lcoe, "-", color=COLORS[tech_name], lw=2,
                    label=f"{tech_name} utility")
            ax.plot(lc_data["year"], roof_lcoe, "--", color=COLORS[tech_name], lw=2,
                    alpha=0.7, label=f"{tech_name} rooftop")
        ax.set_title(f"{city}", fontweight="bold")
        ax.set_xlabel('year'); ax.set_ylabel('LCOE (cents/kWh)')
        ax.grid(alpha=0.3); ax.legend(loc="upper right", fontsize=8)

    fig.suptitle('rooftop (Area constraint) vs utility (kWp Constraint): Different technology-choice criteria',
                 fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig("outputs/figures/41_physics_rooftop_vs_utility.png", dpi=130,
                bbox_inches="tight")
    plt.close(fig)


In [ ]:
%%plot_module scripts.portfolio_physics
from pvsim.labels import label as _text_label
def main():
    viz.setup()
    os.makedirs("outputs/figures", exist_ok=True)

    yield_df = yield_physics_year0()
    yield_df.to_csv("outputs/portfolio_physics_yield.csv", index=False,
                    encoding="utf-8-sig")

    lcoe_df, cap_path = build_city_year_lcoe(yield_df)
    lcoe_df.to_csv("outputs/portfolio_physics_lcoe.csv", index=False,
                   encoding="utf-8-sig")

    cross_df = find_substitution_year(lcoe_df)
    cross_df.to_csv("outputs/portfolio_physics_crossover.csv", index=False,
                    encoding="utf-8-sig")

    print('\n[4/4] Plotting...')
    plot_fig37_temperature_advantage(yield_df)
    plot_fig38_spectral_mismatch(yield_df)
    plot_fig39_city_lcoe_evolution(lcoe_df)
    plot_fig40_substitution_map(cross_df, yield_df, lcoe_df)
    plot_fig41_rooftop_vs_utility(yield_df, lcoe_df)


    print('\n===== Technology substitution results =====')
    print(f'\nAnnual operating temperature ranking (c-Si modules):')
    t_order = yield_df[yield_df["tech"] == _text_label('tech_csi')].sort_values("tcell_weighted")
    for _, r in t_order.iterrows():
        print(f"  {r['city']:5s} : Tcell={r['tcell_weighted']:.1f}°C, "
              f"yield={r['yield_kwh_per_kwp']:.0f} kWh/kWp")
    print(f'\nPerovskite temperature advantage (yield relative to c-Si):')
    for _, r in t_order.iterrows():
        p = yield_df[(yield_df["city"] == r["city"]) &
                     (yield_df["tech"] == _text_label('tech_perovskite'))].iloc[0]
        adv = (p["yield_kwh_per_kwp"] / r["yield_kwh_per_kwp"] - 1) * 100
        print(f"  {r['city']:5s} : +{adv:>4.1f}%")
    print(f'\nCrossover year (City-level LCOE natural crossover):')
    for _, r in cross_df.iterrows():
        v1 = r[_text_label('perovskite_csi_crossover')]; v2 = r[_text_label('tandem_csi_crossover')]
        print(f"  {r['city']:5s} : Perovskite→c-Si {(v1 if v1 else _text_label('none'))} | Tandem→c-Si {(v2 if v2 else _text_label('none'))}")
    print('\nFigure: 37/38/39/40/41 Saved to outputs/figures/')


In [ ]:
run_figure_module('scripts.portfolio_physics', RUNTIME)


## Technology substitution and scenarios


In [ ]:
%%plot_module scripts.fig_substitution_validation
"""Technology-share calibration and scenario distributions."""

from pvsim.labels import label as _text_label

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pvsim import viz
from pvsim.economic_priors import (
    INITIAL_GLOBAL_DEPLOYMENT_GW,
    INITIAL_SYSTEM_CAPEX_USD_W,
    MC_DRAWS,
    MC_RANDOM_SEED,
    TECH_CSI,
    TECH_PEROVSKITE,
    TECH_TANDEM,
    sample_mc_inputs,
)
from pvsim.economics import (
    DISCOUNT_RATE,
    MODERN_CSI_DURABILITY,
    OPEX_FRACTION_OF_CAPEX,
    TANDEM_DURABILITY,
    discounted_lcoe,
    perovskite_durability,
)
from pvsim.provinces import PROVINCE_PV_2024_GW
from pvsim.policy_data import china_pv_target_gw

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

COLORS = {_text_label('tech_csi'): "#1f6fb2", _text_label('tech_perovskite'): "#e2641e", _text_label('tech_tandem'): "#2a9d4a"}

# ============================================================

# ============================================================
HIST_YEARS = np.arange(2015, 2024)

MONO_SHARE_OBS = np.array([0.24, 0.28, 0.35, 0.46, 0.62, 0.84, 0.92, 0.96, 0.98])

MULTI_PRICE = np.array([0.57, 0.48, 0.37, 0.28, 0.23, 0.21, 0.25, 0.26, 0.18])
MONO_PRICE  = np.array([0.67, 0.55, 0.41, 0.30, 0.24, 0.21, 0.24, 0.25, 0.17])

MULTI_EFF = np.array([15.8, 16.3, 17.0, 17.6, 18.2, 18.8, 19.2, 19.5, 19.8])
MONO_EFF  = np.array([16.5, 17.2, 18.3, 19.2, 20.3, 21.2, 22.0, 22.8, 23.3])

AREA_BOS = 45.0
FIXED_BOS = 0.22
DISCOUNT = DISCOUNT_RATE
OPEX = OPEX_FRACTION_OF_CAPEX
YIELD_REF = 1400.0


In [ ]:
%%plot_module scripts.fig_substitution_validation
def system_lcoe(module_price, eff_pct, life=25, deg=0.007):

    w_per_m2 = eff_pct * 10.0   # eff% → W/m^2 (STC 1000 W/m^2)
    sys_price = module_price + AREA_BOS / w_per_m2 + FIXED_BOS   # $/W
    crf = DISCOUNT * (1+DISCOUNT)**life / ((1+DISCOUNT)**life - 1)
    ann = YIELD_REF * (1 - life*deg/2) / 1000.0
    return sys_price * (crf + OPEX) / ann * 100


In [ ]:
%%plot_module scripts.fig_substitution_validation
from pvsim.labels import label as _text_label
def calibrate_T():

    lc_multi = np.array([system_lcoe(MULTI_PRICE[i], MULTI_EFF[i])
                         for i in range(len(HIST_YEARS))])
    lc_mono = np.array([system_lcoe(MONO_PRICE[i], MONO_EFF[i])
                        for i in range(len(HIST_YEARS))])
    best_T, best_rmse, best_pred = None, 1e9, None
    for T in np.arange(0.05, 1.5, 0.01):
        w_mono = np.exp(-lc_mono / T)
        w_multi = np.exp(-lc_multi / T)
        pred = w_mono / (w_mono + w_multi)
        rmse = np.sqrt(np.mean((pred - MONO_SHARE_OBS) ** 2))
        if rmse < best_rmse:
            best_rmse, best_T, best_pred = rmse, T, pred
    ss_res = np.sum((best_pred - MONO_SHARE_OBS) ** 2)
    ss_tot = np.sum((MONO_SHARE_OBS - MONO_SHARE_OBS.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    return best_T, best_rmse, r2, best_pred, lc_multi, lc_mono


# ============================================================

# ============================================================
YEARS = np.arange(2025, 2051)
NY = len(YEARS)
TECHS = [_text_label('tech_csi'), _text_label('tech_perovskite'), _text_label('tech_tandem')]
CAPEX_0 = dict(INITIAL_SYSTEM_CAPEX_USD_W)
Q_0 = dict(INITIAL_GLOBAL_DEPLOYMENT_GW)
INIT_FLEET = 887.0
ROW = 0.4


In [ ]:
%%plot_module scripts.fig_substitution_validation
def nat_yield():
    df = pd.read_csv("outputs/province_physics_yield.csv", encoding="utf-8-sig")
    tot = sum(PROVINCE_PV_2024_GW.values())
    return {t: sum(r["yield_kwh_per_kwp"] * PROVINCE_PV_2024_GW.get(r["province"], 0) / tot
                   for _, r in df[df["tech"] == t].iterrows()) for t in TECHS}


In [ ]:
%%plot_module scripts.fig_substitution_validation
from pvsim.labels import label as _text_label
def max_new(tech, yr):
    if tech == _text_label('tech_csi'): return 350
    if tech == _text_label('tech_perovskite'): return max(0, min(120, (yr-2024)*12))
    if tech == _text_label('tech_tandem'): return max(0, min(180, (yr-2028)*18))
    return 0


In [ ]:
%%plot_module scripts.fig_substitution_validation
def init_present(yr):
    if yr < 2043: return INIT_FLEET
    if yr > 2049: return 0.0
    return INIT_FLEET * (1 - (yr-2043)/6)




In [ ]:
%%plot_module scripts.fig_substitution_validation
from pvsim.labels import label as _text_label
def global_new_schedule():
    a = np.arange(NY)
    return {_text_label('tech_csi'): np.linspace(80, 30, NY) + np.linspace(0, 20, NY),
            _text_label('tech_perovskite'): np.minimum(a*4+5, 100.0),
            _text_label('tech_tandem'): np.maximum(0, np.minimum(a*3-12, 80.0))}


In [ ]:
%%plot_module scripts.fig_substitution_validation
def capex_and_lcoe(
    yields,
    LR,
    floor,
    breakthrough,
    life_final=25.0,
    *,
    return_capex=False,
):

    B = {k: -np.log2(1-v) for k, v in LR.items()}
    sched = global_new_schedule()
    cum = {k: Q_0[k] for k in TECHS}; cap = dict(CAPEX_0)
    lcoe = {k: np.zeros(NY) for k in TECHS}
    capex = {k: np.zeros(NY) for k in TECHS}
    for i, yr in enumerate(YEARS):
        for k in TECHS:
            raw = max(floor[k], CAPEX_0[k]*(max(cum[k],0.1)/Q_0[k])**(-B[k]))
            cap[k] = raw if i == 0 else max(cap[k]*0.88, raw, floor[k])
            capex[k][i] = cap[k]
            if k == TECH_PEROVSKITE:
                durability = perovskite_durability(yr, breakthrough, life_final)
            elif k == TECH_CSI:
                durability = MODERN_CSI_DURABILITY
            else:
                durability = TANDEM_DURABILITY
            lcoe[k][i] = discounted_lcoe(
                cap[k],
                yields[k],
                durability.lifetime_years,
                durability.degradation_rate,
                durability.burn_in_loss,
            ) * 100.0
        for k in TECHS:
            cum[k] += sched[k][i]*(1+ROW)
    return (lcoe, capex) if return_capex else lcoe


In [ ]:
%%plot_module scripts.fig_substitution_validation
def forward(yields, LR, floor, breakthrough, T, life_final=25.0):

    lcoe = capex_and_lcoe(yields, LR, floor, breakthrough, life_final)
    _, target = china_pv_target_gw(); target = target[1:]
    deploy = {k: np.zeros(NY) for k in TECHS}
    for i, yr in enumerate(YEARS):
        op = {}
        for k in TECHS:
            operating = 0.0
            for j in range(i):
                if k == TECH_PEROVSKITE:
                    installed_life = perovskite_durability(
                        YEARS[j], breakthrough, life_final,
                    ).lifetime_years
                elif k == TECH_CSI:
                    installed_life = MODERN_CSI_DURABILITY.lifetime_years
                else:
                    installed_life = TANDEM_DURABILITY.lifetime_years
                if yr - YEARS[j] < installed_life:
                    operating += deploy[k][j]
            op[k] = operating
        need = max(0, target[i] - sum(op.values()) - init_present(yr))
        wts = {k: np.exp(-lcoe[k][i]/T) for k in TECHS if max_new(k, yr) > 0}
        s = sum(wts.values())
        tg = {k: need*wts[k]/s for k in wts}
        left = 0.0
        for k in TECHS:
            c = min(tg.get(k,0)+left, max_new(k, yr)); deploy[k][i] = c
            left += tg.get(k,0)-c
        for k in TECHS:
            if left > 0.5 and deploy[k][i] < max_new(k, yr):
                e = min(left, max_new(k, yr)-deploy[k][i]); deploy[k][i] += e; left -= e
    cross = next((
        int(YEARS[i]) for i in range(NY)
        if lcoe[TECH_PEROVSKITE][i] < lcoe[TECH_CSI][i]
    ), 2051)
    share = {k: deploy[k]/np.maximum(sum(deploy[t] for t in TECHS), 1e-9) for k in TECHS}
    return share, cross


In [ ]:
%%plot_module scripts.fig_substitution_validation
from pvsim.labels import label as _text_label
def main():
    viz.setup_en()
    os.makedirs("outputs/figures", exist_ok=True)

    print("[A] historical analog calibration (multi->mono)...")
    T_fit, rmse, r2, pred, lc_multi, lc_mono = calibrate_T()
    print(f"  fitted T={T_fit:.2f} cents/kWh, RMSE={rmse*100:.1f}%, R2={r2:.3f}")

    print(f"[B] Monte Carlo {MC_DRAWS} runs...")
    yields = nat_yield()
    rng = np.random.default_rng(MC_RANDOM_SEED)
    cross_samples = []
    perov_share_2050 = []
    share_paths = {k: [] for k in TECHS}
    for _ in range(MC_DRAWS):
        LR, floor, bt, life_final, T = sample_mc_inputs(rng, T_fit)
        share, cross = forward(yields, LR, floor, bt, T, life_final)
        cross_samples.append(cross)
        perov_share_2050.append(share[_text_label('tech_perovskite')][-1]*100)
        for k in TECHS:
            share_paths[k].append(share[k]*100)
    cross_samples = np.array(cross_samples)
    for k in TECHS:
        share_paths[k] = np.array(share_paths[k])


    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=300)


    ax = axes[0]
    ax.plot(HIST_YEARS, MONO_SHARE_OBS*100, "o", color="black", ms=8,
            label="Mono share, observed (ITRPV)", zorder=4)
    ax.plot(HIST_YEARS, pred*100, "-", color="#e2641e", lw=2,
            label=f"softmax model (T={T_fit:.2f})", zorder=3)
    ax.set_xlabel("Year", fontsize=9); ax.set_ylabel("Mono-Si market share (%)", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.set_title("(a) Historical calibration: multi->mono", fontsize=11,
                 fontweight="bold", loc="left", pad=3)
    ax.text(0.05, 0.74, f"R$^2$ = {r2:.3f}\nRMSE = {rmse*100:.1f}%\n"
            "$\\rightarrow$ mechanism credible",
            transform=ax.transAxes, fontsize=8.5,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.5))
    ax.legend(fontsize=7.5, loc="lower right", frameon=False)
    ax.grid(alpha=0.25, lw=0.3)

    # (b) Monte Carlo share fan
    ax = axes[1]
    EN = {_text_label('tech_csi'): "c-Si", _text_label('tech_perovskite'): "Perovskite", _text_label('tech_tandem'): "Tandem"}
    for k in TECHS:
        p10 = np.percentile(share_paths[k], 10, axis=0)
        p50 = np.percentile(share_paths[k], 50, axis=0)
        p90 = np.percentile(share_paths[k], 90, axis=0)
        ax.fill_between(YEARS, p10, p90, color=COLORS[k], alpha=0.2, linewidth=0)
        ax.plot(YEARS, p50, color=COLORS[k], lw=2, label=EN[k])
    ax.set_xlabel("Year", fontsize=9); ax.set_ylabel("Annual new-build share (%)", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.set_title(f"(b) Monte Carlo ({MC_DRAWS} runs): P10-P90 band", fontsize=11,
                 fontweight="bold", loc="left", pad=3)
    ax.legend(fontsize=7.5, loc="center right", frameon=False)
    ax.set_xlim(2025, 2050); ax.set_ylim(0, 100); ax.grid(alpha=0.25, lw=0.3)

    # (c) perovskite-beats-c-Si crossover-year distribution
    ax = axes[2]
    valid = cross_samples[cross_samples <= 2050]
    ax.hist(valid, bins=range(2026, 2046), color="#e2641e", alpha=0.8,
            edgecolor="black", linewidth=0.4)
    p10c, p50c, p90c = np.percentile(valid, [10, 50, 90]) if len(valid) else (0, 0, 0)
    for v, lab, c in [(p10c, "P10", "#2a9d4a"), (p50c, "P50", "black"),
                      (p90c, "P90", "#d62728")]:
        ax.axvline(v, color=c, ls="--", lw=1.2)
        ax.text(v, ax.get_ylim()[1]*0.92, f"{lab}\n{v:.0f}", fontsize=7,
                ha="center", color=c, fontweight="bold")
    ax.set_xlabel("Perovskite-beats-c-Si crossover year", fontsize=9)
    ax.set_ylabel("Frequency (/1000)", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.set_title("(c) Crossover-year uncertainty", fontsize=11, fontweight="bold",
                 loc="left", pad=3)
    ax.grid(alpha=0.25, lw=0.3, axis="y")

    fig.suptitle("Fig 6 — Substitution model: historical calibration (R$^2$=%.2f) + "
                 "Monte Carlo uncertainty (crossover P10-P90: %d-%d)"
                 % (r2, p10c, p90c), fontsize=12, fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig("outputs/figures/MainFig6_substitution_validation.png", dpi=300,
                bbox_inches="tight")
    fig.savefig("outputs/figures/MainFig6_substitution_validation.pdf",
                bbox_inches="tight")
    plt.close(fig)
    print(f"\nMain Fig 6 saved.")
    print(f"  crossover year P10/P50/P90 = {p10c:.0f}/{p50c:.0f}/{p90c:.0f}")
    print(f"  perovskite 2050 share P10/P50/P90 = "
          f"{np.percentile(perov_share_2050,10):.0f}/"
          f"{np.percentile(perov_share_2050,50):.0f}/"
          f"{np.percentile(perov_share_2050,90):.0f}%")
    print(f"  never-substitutes (>2050) fraction: {np.mean(cross_samples>2050)*100:.1f}%")


In [ ]:
run_figure_module('scripts.fig_substitution_validation', RUNTIME)


## Outdoor measurement validation


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
"""Paired outdoor-device measurements and figures."""

from __future__ import annotations

import csv
import hashlib
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pvsim.cell import operating_point
from pvsim.materials import CSI_MODERN, PEROVSKITE


ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = ROOT / "data" / "external" / "jaramillo_montoya_2018"
RAW_DIR = DATA_DIR / "raw"
OUT_PAIRS = ROOT / "outputs" / "si_jaramillo_montoya_outdoor_pairs.csv"
OUT_BINS = ROOT / "outputs" / "si_jaramillo_montoya_outdoor_bins.csv"
OUT_SUMMARY = ROOT / "outputs" / "si_jaramillo_montoya_outdoor_summary.csv"
OUT_FIGURE = ROOT / "outputs" / "figures" / "SI_external_outdoor_validation.png"
OUT_FIGURE_PDF = ROOT / "outputs" / "figures" / "SI_external_outdoor_validation.pdf"

DATASET_DOI = "10.17632/9jx8mdh8xd.1"
ARTICLE_DOI = "10.1016/j.solmat.2018.10.018"

RAW_HASHES = {
    "Datos_Atmos_01_2018.csv": "c0c7e84aaa8c877acb26bdfd9a07f3ecdab409845c491c86b6422037700ebbb1",
    "Datos_Atmos_02_2018.csv": "6d459dd742cb2deb34572a1ad88520c07a0e142d5a43e396cd3272b44ecfa887",
    "Datos_Atmos_03_2018.csv": "f4b777a7759ddd28d59ab886ed5a4bdd4c1dbd5f1c2f68ffbfdc4ee02c70771e",
    "Datos_Atmos_04_2018.csv": "a49b9d81c96e4f3c183333272d81334c25be9d31ab29aa86ed0b6527d6d4d1ab",
    "PSM17_I-V_data.xls": "fb4a0da6dd2d0d07de67cfccaa73193b23f29a9b6d9e8c2c4bf792a6b97a9945",
    "PSM50_IV.csv": "54a04096dde12f107d88987c1b0316564f27a5001a933d511af18379b4473040",
    "Sharp_I-V_data.xls": "0f89206965754494c40d7a51dcc349c02e447a78dc70e8ba79835e5df649ba45",
    "Supporting information_28_09_2018.docx": "9b6de985e5d1c585904319452db128e8e15cea8d2ec31fdf61715f107de16ac7",
}

SESSIONS = {
    "PSM17": {
        "filename": "PSM17_I-V_data.xls",
        "area_cm2": 17.0,
        "pmax_scale_to_w": 0.001,
        "pmax_max_w": 0.12992,
        "voc_max_v": 7.1771,
    },
    "PSM50": {
        "filename": "PSM50_IV.csv",
        "area_cm2": 50.0,
        "pmax_scale_to_w": 0.001,
        "pmax_max_w": 0.18623,
        "voc_max_v": 6.1569,
    },
}

IRRADIANCE_EDGES = np.array([150.0, 300.0, 500.0, 700.0, 900.0, 1100.0, 1200.0001])


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def audit_raw_files(raw_dir: Path = RAW_DIR) -> list[dict[str, str | int | bool]]:
    rows: list[dict[str, str | int | bool]] = []
    for filename, expected in RAW_HASHES.items():
        path = raw_dir / filename
        actual = _sha256(path) if path.exists() else ""
        rows.append(
            {
                "filename": filename,
                "exists": path.exists(),
                "bytes": path.stat().st_size if path.exists() else 0,
                "sha256_expected": expected,
                "sha256_actual": actual,
                "sha256_ok": actual == expected,
            }
        )
    failures = [str(row["filename"]) for row in rows if not row["sha256_ok"]]
    if failures:
        raise ValueError(f"Missing or altered raw files: {failures}")
    return rows


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def load_atmosphere(raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    frames = []
    for path in sorted(raw_dir.glob("Datos_Atmos_*.csv")):
        frame = pd.read_csv(path, sep=";", skiprows=4)
        frame["atmosphere_time"] = pd.to_datetime(
            frame["Fecha"].astype(str) + " " + frame["Hora"].astype(str),
            format="%d/%m/%y %H:%M:%S",
            errors="raise",
        )
        numeric = [
            "Panel_temperature_1",
            "Panel_temperature_2",
            "Ambient_temperature_1",
            "Ambient_temperature_2",
            "Irradiance_1",
            "Irradiance_2",
        ]
        for column in numeric:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frames.append(frame[["atmosphere_time", *numeric]])
    if len(frames) != 4:
        raise ValueError(f"Expected four atmospheric files, got {len(frames)}")
    out = (
        pd.concat(frames, ignore_index=True)
        .sort_values("atmosphere_time")
        .drop_duplicates("atmosphere_time")
    )
    out["irradiance_w_m2"] = out[["Irradiance_1", "Irradiance_2"]].mean(axis=1)
    out["ambient_temperature_c"] = out[
        ["Ambient_temperature_1", "Ambient_temperature_2"]
    ].mean(axis=1)
    return out.reset_index(drop=True)


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def _read_psm50_csv(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, sep=";", quoting=csv.QUOTE_NONE, dtype=str)
    frame.columns = [str(column).strip().strip('"') for column in frame.columns]
    for column in frame.columns:
        frame[column] = frame[column].astype(str).str.strip().str.strip('"')
    return frame


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def load_device(filename: str, raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    path = raw_dir / filename
    if path.suffix.lower() == ".xls":
        frame = pd.read_excel(path, engine="xlrd")
    else:
        frame = _read_psm50_csv(path)
    frame = frame.rename(
        columns={
            "Pmax(W)": "pmax_raw",
            "Pmax": "pmax_raw",
            "Voc(V)": "voc_v",
            "Voc": "voc_v",
            "Isc(A)": "isc_raw",
            "Isc": "isc_raw",
        }
    )
    frame["measurement_time"] = pd.to_datetime(frame["Date"], errors="raise")
    for column in ["pmax_raw", "voc_v", "isc_raw"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame[["measurement_time", "pmax_raw", "voc_v", "isc_raw"]].sort_values(
        "measurement_time"
    )


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def build_paired_session(session: str, raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    if session not in SESSIONS:
        raise KeyError(f"Unknown session {session}")
    config = SESSIONS[session]
    psm = load_device(str(config["filename"]), raw_dir).rename(
        columns={
            "measurement_time": "psm_time",
            "pmax_raw": "psm_pmax_raw",
            "voc_v": "psm_voc_v",
            "isc_raw": "psm_isc_raw",
        }
    )
    silicon = load_device("Sharp_I-V_data.xls", raw_dir).rename(
        columns={
            "measurement_time": "silicon_time",
            "pmax_raw": "silicon_pmax_w",
            "voc_v": "silicon_voc_v",
            "isc_raw": "silicon_isc_raw",
        }
    )
    atmosphere = load_atmosphere(raw_dir)
    paired = pd.merge_asof(
        psm,
        silicon,
        left_on="psm_time",
        right_on="silicon_time",
        direction="nearest",
        tolerance=pd.Timedelta(seconds=90),
    )
    paired = pd.merge_asof(
        paired.sort_values("psm_time"),
        atmosphere,
        left_on="psm_time",
        right_on="atmosphere_time",
        direction="nearest",
        tolerance=pd.Timedelta(seconds=90),
    )
    paired["session"] = session
    paired["psm_area_cm2"] = float(config["area_cm2"])
    paired["psm_pmax_w"] = paired["psm_pmax_raw"] * float(config["pmax_scale_to_w"])
    paired["psm_match_seconds"] = (
        paired["psm_time"] - paired["silicon_time"]
    ).dt.total_seconds().abs()
    paired["atmosphere_match_seconds"] = (
        paired["psm_time"] - paired["atmosphere_time"]
    ).dt.total_seconds().abs()

    required = [
        "psm_time",
        "silicon_time",
        "atmosphere_time",
        "psm_pmax_w",
        "psm_voc_v",
        "silicon_pmax_w",
        "silicon_voc_v",
        "irradiance_w_m2",
        "Irradiance_1",
        "Irradiance_2",
        "Panel_temperature_1",
        "Panel_temperature_2",
        "ambient_temperature_c",
    ]
    paired = paired.dropna(subset=required).copy()
    sensor_tolerance = np.maximum(75.0, 0.15 * paired["irradiance_w_m2"])
    keep = (
        paired["irradiance_w_m2"].between(150.0, 1200.0)
        & ((paired["Irradiance_1"] - paired["Irradiance_2"]).abs() <= sensor_tolerance)
        & paired["psm_pmax_w"].between(0.0, float(config["pmax_max_w"]), inclusive="right")
        & paired["psm_voc_v"].between(0.2, float(config["voc_max_v"]), inclusive="right")
        & paired["silicon_pmax_w"].between(0.0, 315.10, inclusive="right")
        & paired["silicon_voc_v"].between(20.0, 38.455, inclusive="right")
        & paired["Panel_temperature_1"].between(0.0, 85.0)
        & paired["Panel_temperature_2"].between(0.0, 85.0)
    )
    paired = paired.loc[keep].copy()
    start = paired["psm_time"].min()
    paired["elapsed_day"] = (paired["psm_time"] - start).dt.total_seconds() / 86400.0
    paired["calendar_day"] = paired["psm_time"].dt.strftime("%Y-%m-%d")

    high = paired["irradiance_w_m2"].between(900.0, 1100.0)
    if int(high.sum()) < 30:
        raise ValueError(f"Insufficient high-irradiance reference rows for {session}")
    psm_ref = float(
        np.median(paired.loc[high, "psm_pmax_w"] / paired.loc[high, "irradiance_w_m2"])
    )
    silicon_ref = float(
        np.median(
            paired.loc[high, "silicon_pmax_w"] / paired.loc[high, "irradiance_w_m2"]
        )
    )
    paired["psm_power_per_irradiance_rel"] = (
        paired["psm_pmax_w"] / paired["irradiance_w_m2"] / psm_ref
    )
    paired["silicon_power_per_irradiance_rel"] = (
        paired["silicon_pmax_w"] / paired["irradiance_w_m2"] / silicon_ref
    )
    paired["paired_relative_power_ratio"] = (
        paired["psm_power_per_irradiance_rel"]
        / paired["silicon_power_per_irradiance_rel"]
    )
    return paired.reset_index(drop=True)


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def build_paired_data(raw_dir: Path = RAW_DIR) -> pd.DataFrame:
    return pd.concat(
        [build_paired_session(session, raw_dir) for session in SESSIONS],
        ignore_index=True,
    )


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def _ols_coefficient(frame: pd.DataFrame, technology: str) -> float:
    high = frame[frame["irradiance_w_m2"].between(800.0, 1100.0)]
    if technology == "perovskite":
        response = np.log(high["psm_voc_v"].to_numpy(float))
        temperature = high["Panel_temperature_1"].to_numpy(float)
    elif technology == "silicon":
        response = np.log(high["silicon_voc_v"].to_numpy(float))
        temperature = high["Panel_temperature_2"].to_numpy(float)
    else:
        raise KeyError(technology)
    session_dummy = (high["session"] == "PSM50").to_numpy(float)
    design = np.column_stack(
        [
            np.ones(len(high)),
            np.log(high["irradiance_w_m2"].to_numpy(float) / 1000.0),
            temperature - 25.0,
            high["elapsed_day"].to_numpy(float),
            session_dummy,
        ]
    )
    coefficient = np.linalg.lstsq(design, response, rcond=None)[0]
    return float(100.0 * coefficient[2])


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def _ratio_irradiance_slope(frame: pd.DataFrame) -> float:
    response = np.log(frame["paired_relative_power_ratio"].to_numpy(float))
    session_dummy = (frame["session"] == "PSM50").to_numpy(float)
    design = np.column_stack(
        [
            np.ones(len(frame)),
            np.log(frame["irradiance_w_m2"].to_numpy(float) / 1000.0),
            frame["ambient_temperature_c"].to_numpy(float) - 25.0,
            frame["elapsed_day"].to_numpy(float),
            session_dummy,
        ]
    )
    coefficient = np.linalg.lstsq(design, response, rcond=None)[0]
    return float(coefficient[1])


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def _cluster_bootstrap(
    frame: pd.DataFrame,
    estimator: Callable[[pd.DataFrame], float],
    n_boot: int,
    seed: int,
) -> tuple[float, float]:
    if n_boot <= 0:
        return float("nan"), float("nan")
    clusters = frame["calendar_day"].drop_duplicates().to_numpy()
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(n_boot):
        sampled = rng.choice(clusters, size=len(clusters), replace=True)
        blocks = [frame.loc[frame["calendar_day"] == day] for day in sampled]
        sample = pd.concat(blocks, ignore_index=True)
        try:
            value = estimator(sample)
        except (ValueError, np.linalg.LinAlgError):
            continue
        if np.isfinite(value):
            estimates.append(value)
    if len(estimates) < max(50, n_boot // 4):
        raise ValueError("Too few valid day-block bootstrap replicates")
    return tuple(float(value) for value in np.percentile(estimates, [2.5, 97.5]))


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def _model_voc_temperature_coefficient(technology) -> float:
    low = operating_point(
        technology, 1000.0, 20.0, ns=technology.cells_in_series, npts=120
    ).voc
    high = operating_point(
        technology, 1000.0, 30.0, ns=technology.cells_in_series, npts=120
    ).voc
    return float(100.0 * (np.log(high) - np.log(low)) / 10.0)


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def build_bins(pairs: pd.DataFrame, n_boot: int = 1000) -> pd.DataFrame:
    rows = []
    labels = ["150-300", "300-500", "500-700", "700-900", "900-1100", "1100-1200"]
    work = pairs.copy()
    work["irradiance_bin"] = pd.cut(
        work["irradiance_w_m2"],
        IRRADIANCE_EDGES,
        labels=labels,
        right=False,
        include_lowest=True,
    )
    for session in SESSIONS:
        session_rows = work[work["session"] == session]
        for index, label in enumerate(labels):
            group = session_rows[session_rows["irradiance_bin"] == label]
            if group.empty:
                continue
            estimator = lambda sample: float(
                np.median(sample["paired_relative_power_ratio"].to_numpy(float))
            )
            ci_low, ci_high = _cluster_bootstrap(
                group, estimator, n_boot=n_boot, seed=8100 + index + 100 * len(rows)
            )
            rows.append(
                {
                    "session": session,
                    "irradiance_bin_w_m2": label,
                    "n_pairs": len(group),
                    "n_days": group["calendar_day"].nunique(),
                    "median_irradiance_w_m2": float(group["irradiance_w_m2"].median()),
                    "median_ambient_temperature_c": float(
                        group["ambient_temperature_c"].median()
                    ),
                    "median_relative_power_ratio": estimator(group),
                    "ci_low": ci_low,
                    "ci_high": ci_high,
                }
            )
    return pd.DataFrame(rows)


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def build_summary(pairs: pd.DataFrame, n_boot: int = 1000) -> pd.DataFrame:
    perov = _ols_coefficient(pairs, "perovskite")
    silicon = _ols_coefficient(pairs, "silicon")
    differential = perov - silicon
    perov_ci = _cluster_bootstrap(
        pairs, lambda sample: _ols_coefficient(sample, "perovskite"), n_boot, 20260720
    )
    silicon_ci = _cluster_bootstrap(
        pairs, lambda sample: _ols_coefficient(sample, "silicon"), n_boot, 20260721
    )
    differential_ci = _cluster_bootstrap(
        pairs,
        lambda sample: _ols_coefficient(sample, "perovskite")
        - _ols_coefficient(sample, "silicon"),
        n_boot,
        20260722,
    )
    ratio_slope = _ratio_irradiance_slope(pairs)
    ratio_ci = _cluster_bootstrap(pairs, _ratio_irradiance_slope, n_boot, 20260723)
    model_perov = _model_voc_temperature_coefficient(PEROVSKITE)
    model_silicon = _model_voc_temperature_coefficient(CSI_MODERN)
    model_differential = model_perov - model_silicon

    rows = [
        {
            "metric": "paired_rows_total",
            "value": float(len(pairs)),
            "ci_low": np.nan,
            "ci_high": np.nan,
            "unit": "minute pairs",
            "interpretation": "quality-controlled co-located observations",
        },
        {
            "metric": "paired_days_total",
            "value": float(pairs["calendar_day"].nunique()),
            "ci_low": np.nan,
            "ci_high": np.nan,
            "unit": "days",
            "interpretation": "independent day clusters used for uncertainty",
        },
        {
            "metric": "observed_perovskite_voc_temperature_coefficient",
            "value": perov,
            "ci_low": perov_ci[0],
            "ci_high": perov_ci[1],
            "unit": "percent per C",
            "interpretation": "pooled PSM17 and PSM50 high-irradiance field estimate",
        },
        {
            "metric": "observed_silicon_voc_temperature_coefficient",
            "value": silicon,
            "ci_low": silicon_ci[0],
            "ci_high": silicon_ci[1],
            "unit": "percent per C",
            "interpretation": "co-located Sharp panel high-irradiance field estimate",
        },
        {
            "metric": "observed_voc_temperature_coefficient_differential",
            "value": differential,
            "ci_low": differential_ci[0],
            "ci_high": differential_ci[1],
            "unit": "percentage points per C",
            "interpretation": "positive means perovskite voltage is less temperature-sensitive",
        },
        {
            "metric": "model_perovskite_voc_temperature_coefficient",
            "value": model_perov,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "unit": "percent per C",
            "interpretation": "current single-diode model at 1000 W per square metre",
        },
        {
            "metric": "model_silicon_voc_temperature_coefficient",
            "value": model_silicon,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "unit": "percent per C",
            "interpretation": "current modern-silicon model at 1000 W per square metre",
        },
        {
            "metric": "model_voc_temperature_coefficient_differential",
            "value": model_differential,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "unit": "percentage points per C",
            "interpretation": "model thermal-ranking contrast",
        },
        {
            "metric": "paired_power_ratio_log_irradiance_slope",
            "value": ratio_slope,
            "ci_low": ratio_ci[0],
            "ci_high": ratio_ci[1],
            "unit": "log ratio per log irradiance",
            "interpretation": "positive means this dataset does not support instantaneous low-light superiority",
        },
    ]
    for session in SESSIONS:
        session_rows = pairs[pairs["session"] == session]
        rows.append(
            {
                "metric": f"{session.lower()}_paired_rows",
                "value": float(len(session_rows)),
                "ci_low": np.nan,
                "ci_high": np.nan,
                "unit": "minute pairs",
                "interpretation": f"quality-controlled {session} and silicon pairs",
            }
        )
    return pd.DataFrame(rows)


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def _summary_value(summary: pd.DataFrame, metric: str, column: str = "value") -> float:
    return float(summary.loc[summary["metric"] == metric, column].iloc[0])


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def write_figure(bins: pd.DataFrame, summary: pd.DataFrame) -> None:
    OUT_FIGURE.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2), constrained_layout=True)
    colors = {"PSM17": "#087E8B", "PSM50": "#D1495B"}
    for session, group in bins.groupby("session", sort=False):
        group = group[group["n_pairs"] >= 30]
        group = group.sort_values("median_irradiance_w_m2")
        x = group["median_irradiance_w_m2"].to_numpy(float)
        y = group["median_relative_power_ratio"].to_numpy(float)
        low = group["ci_low"].to_numpy(float)
        high = group["ci_high"].to_numpy(float)
        axes[0].plot(x, y, marker="o", linewidth=1.8, color=colors[session], label=session)
        axes[0].fill_between(x, low, high, color=colors[session], alpha=0.15, linewidth=0)
    axes[0].axhline(1.0, color="#4A4A4A", linestyle="--", linewidth=1)
    axes[0].set_xlabel("Irradiance bin median, W m$^{-2}$")
    axes[0].set_ylabel("Relative PSM-to-silicon power response")
    axes[0].set_title("a  Co-located irradiance response", loc="left", fontweight="bold")
    axes[0].legend(frameon=False)
    axes[0].grid(alpha=0.2)

    observed = [
        _summary_value(summary, "observed_perovskite_voc_temperature_coefficient"),
        _summary_value(summary, "observed_silicon_voc_temperature_coefficient"),
        _summary_value(summary, "observed_voc_temperature_coefficient_differential"),
    ]
    low = [
        _summary_value(summary, "observed_perovskite_voc_temperature_coefficient", "ci_low"),
        _summary_value(summary, "observed_silicon_voc_temperature_coefficient", "ci_low"),
        _summary_value(
            summary, "observed_voc_temperature_coefficient_differential", "ci_low"
        ),
    ]
    high = [
        _summary_value(summary, "observed_perovskite_voc_temperature_coefficient", "ci_high"),
        _summary_value(summary, "observed_silicon_voc_temperature_coefficient", "ci_high"),
        _summary_value(
            summary, "observed_voc_temperature_coefficient_differential", "ci_high"
        ),
    ]
    modeled = [
        _summary_value(summary, "model_perovskite_voc_temperature_coefficient"),
        _summary_value(summary, "model_silicon_voc_temperature_coefficient"),
        _summary_value(summary, "model_voc_temperature_coefficient_differential"),
    ]
    xpos = np.array([0.0, 1.0, 2.0])
    errors = np.vstack([np.array(observed) - np.array(low), np.array(high) - np.array(observed)])
    axes[1].errorbar(
        xpos - 0.08,
        observed,
        yerr=errors,
        fmt="o",
        markersize=7,
        capsize=4,
        color="#087E8B",
        label="Outdoor estimate",
    )
    axes[1].scatter(xpos + 0.08, modeled, marker="s", s=48, color="#D1495B", label="Current model")
    axes[1].axhline(0.0, color="#4A4A4A", linewidth=0.8)
    axes[1].set_xticks(xpos, ["Perovskite", "Silicon", "Difference"])
    axes[1].set_ylabel(r"$V_{OC}$ temperature coefficient, % $^\circ$C$^{-1}$")
    axes[1].set_title("b  Thermal-response ranking", loc="left", fontweight="bold")
    axes[1].legend(frameon=False, loc="upper left")
    axes[1].grid(axis="y", alpha=0.2)

    fig.savefig(OUT_FIGURE, dpi=300, bbox_inches="tight")
    fig.savefig(OUT_FIGURE_PDF, bbox_inches="tight")
    plt.close(fig)


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def run(n_boot: int = 1000) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    audit_raw_files()
    pairs = build_paired_data()
    bins = build_bins(pairs, n_boot=n_boot)
    summary = build_summary(pairs, n_boot=n_boot)
    OUT_PAIRS.parent.mkdir(parents=True, exist_ok=True)
    pairs.to_csv(OUT_PAIRS, index=False)
    bins.to_csv(OUT_BINS, index=False)
    summary.to_csv(OUT_SUMMARY, index=False)
    write_figure(bins, summary)
    return pairs, bins, summary


In [ ]:
%%plot_module scripts.jaramillo_montoya_outdoor_validation
def main() -> int:
    pairs, _, summary = run()
    differential = _summary_value(
        summary, "observed_voc_temperature_coefficient_differential"
    )
    ratio_slope = _summary_value(summary, "paired_power_ratio_log_irradiance_slope")
    print(f"Outdoor validation written with {len(pairs)} paired minute observations.")
    print(f"Observed thermal differential: {differential:.3f} percentage points per C.")
    print(f"Paired power-ratio log irradiance slope: {ratio_slope:.3f}.")
    return 0


In [ ]:
run_figure_module('scripts.jaramillo_montoya_outdoor_validation', RUNTIME)


## Silicon system sensitivity


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
"""Silicon device and system sensitivity figures."""

from __future__ import annotations

import argparse
import csv
import os
import sys
import traceback
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from dataclasses import replace
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pvsim import viz
from pvsim.cell import operating_point
from pvsim.city_catalog import (
    DEFAULT_CITY_CATALOG,
    CityAnchor,
    load_city_catalog,
)
from pvsim.materials import CSI_MODERN, PEROVSKITE, CellTechnology
from pvsim.system import SystemConfig, simulate
from pvsim import weather as wx


if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[1]
OUT_CSV = ROOT / "outputs" / "si_silicon_baseline_sensitivity.csv"
OUT_SUMMARY = ROOT / "outputs" / "si_silicon_baseline_sensitivity_summary.csv"
OUT_FIGURE = ROOT / "outputs" / "figures" / "SI_silicon_system_sensitivity"
DEFAULT_SHARD_DIR = ROOT / "outputs" / "si_silicon_system_sensitivity_shards"

SCENARIOS = (
    {
        "scenario": "baseline",
        "label": "Baseline",
        "group": "baseline",
        "description": "Modern c-Si baseline; monofacial open-rack; DC/AC 1.2",
    },
    {
        "scenario": "hjt_gamma",
        "label": "HJT-like γ −0.24",
        "group": "device",
        "description": "Isolated c-Si power-temperature response at −0.24 %/C",
        "csi_gamma": -0.24,
    },
    {
        "scenario": "topcon_gamma",
        "label": "TOPCon-like γ −0.28",
        "group": "device",
        "description": "Isolated c-Si power-temperature response at −0.28 %/C",
        "csi_gamma": -0.28,
    },
    {
        "scenario": "perc_gamma",
        "label": "PERC-like γ −0.35",
        "group": "device",
        "description": "Isolated c-Si power-temperature response at −0.35 %/C",
        "csi_gamma": -0.35,
    },
    {
        "scenario": "hot_roof",
        "label": "Hot roof 20/3",
        "group": "mounting",
        "description": "Shared Faiman coefficients u0=20, u1=3",
        "config": {"thermal_u0": 20.0, "thermal_u1": 3.0},
    },
    {
        "scenario": "cool_rack",
        "label": "Cool rack 29/8",
        "group": "mounting",
        "description": "Shared Faiman coefficients u0=29, u1=8",
        "config": {"thermal_u0": 29.0, "thermal_u1": 8.0},
    },
    {
        "scenario": "dcac_11",
        "label": "DC/AC 1.1",
        "group": "inverter",
        "description": "Shared DC/AC ratio 1.1",
        "config": {"dc_ac_ratio": 1.1},
    },
    {
        "scenario": "dcac_13",
        "label": "DC/AC 1.3",
        "group": "inverter",
        "description": "Shared DC/AC ratio 1.3",
        "config": {"dc_ac_ratio": 1.3},
    },
    {
        "scenario": "csi_bifacial",
        "label": "c-Si bifacial 0.70",
        "group": "boundary",
        "description": "Adversarial boundary: c-Si only bifaciality 0.70",
        "csi_config": {"bifaciality_override": 0.70},
    },
)
SCENARIO_ORDER = [item["scenario"] for item in SCENARIOS]
SCENARIO_LOOKUP = {item["scenario"]: item for item in SCENARIOS}


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def _gamma_pmax(tech: CellTechnology) -> float:
    """Modelled Pmax temperature coefficient in percent per C."""
    temps = np.arange(15.0, 66.0, 5.0)
    pmp = np.array([
        operating_point(
            tech, 1000.0, float(temp), ns=tech.cells_in_series,
        ).pmp
        for temp in temps
    ])
    p25 = operating_point(
        tech, 1000.0, 25.0, ns=tech.cells_in_series,
    ).pmp
    slope = np.sum((temps - temps.mean()) * (pmp - pmp.mean()))
    slope /= np.sum((temps - temps.mean()) ** 2)
    return float(slope / p25 * 100.0)


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
@lru_cache(maxsize=None)
def _csi_for_gamma(target_gamma: float) -> CellTechnology:
    """Calibrate Ea so the same modern-c-Si device hits a target gamma."""
    lo, hi = 0.4, 1.4
    flo = _gamma_pmax(replace(CSI_MODERN, Ea_recomb=lo)) - target_gamma
    for _ in range(70):
        mid = 0.5 * (lo + hi)
        fmid = _gamma_pmax(replace(CSI_MODERN, Ea_recomb=mid)) - target_gamma
        if abs(fmid) < 2e-5:
            break
        if (flo < 0) == (fmid < 0):
            lo, flo = mid, fmid
        else:
            hi = mid
    ea = 0.5 * (lo + hi)
    return replace(
        CSI_MODERN,
        Ea_recomb=ea,
        gamma_pmax_lit=target_gamma,
        notes=(
            f"Modern c-Si isolated temperature-response sensitivity; "
            f"target gamma={target_gamma:.2f} %/C."
        ),
    )


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def _weighted_tcell(result: dict) -> float:
    ts = result["timeseries"]
    poa = ts["poa_global"].to_numpy(float)
    mask = poa > 50.0
    if not mask.any():
        return float("nan")
    return float(np.average(
        ts["tcell"].to_numpy(float)[mask],
        weights=poa[mask],
    ))


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def _result_row(
    anchor: CityAnchor,
    weather: pd.DataFrame,
    scenario: dict,
    csi_result: dict,
    perov_result: dict,
    csi_gamma_model: float,
) -> dict:
    csi_yield = float(csi_result["specific_yield"])
    perov_yield = float(perov_result["specific_yield"])
    return {
        "adcode": anchor.adcode,
        "city": anchor.city,
        "city_fullname": anchor.city_fullname,
        "province": anchor.province,
        "province_code": anchor.province_code,
        "lat": anchor.lat,
        "lon": anchor.lon,
        "scenario": scenario["scenario"],
        "scenario_label": scenario["label"],
        "scenario_group": scenario["group"],
        "scenario_description": scenario["description"],
        "csi_gamma_model_pct_per_c": csi_gamma_model,
        "csi_yield_kwh_per_kwp": csi_yield,
        "perovskite_yield_kwh_per_kwp": perov_yield,
        "perovskite_advantage_pct": (perov_yield / csi_yield - 1.0) * 100.0,
        "csi_tcell_weighted_c": _weighted_tcell(csi_result),
        "perovskite_tcell_weighted_c": _weighted_tcell(perov_result),
        "ghi_kwh_m2": float(weather["ghi"].sum() / 1000.0),
        "calculation_basis": "337 administrative-centre full-hourly reruns",
    }


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def _simulate_anchor(
    anchor: CityAnchor,
    npts: int,
    cache_dir: str,
    allow_download: bool,
) -> list[dict]:
    weather = wx.from_pvgis_tmy(
        anchor.lat,
        anchor.lon,
        altitude=anchor.altitude_m,
        name=anchor.cache_key,
        cache_dir=cache_dir,
        allow_download=allow_download,
    )
    base_cfg = SystemConfig(n_modules=20)
    baseline_csi = simulate(CSI_MODERN, weather, base_cfg, npts=npts)
    baseline_perov = simulate(PEROVSKITE, weather, base_cfg, npts=npts)
    baseline_gamma = _gamma_pmax(CSI_MODERN)
    rows = [
        _result_row(
            anchor,
            weather,
            SCENARIO_LOOKUP["baseline"],
            baseline_csi,
            baseline_perov,
            baseline_gamma,
        )
    ]

    for scenario in SCENARIOS[1:]:
        if "csi_gamma" in scenario:
            csi_tech = _csi_for_gamma(float(scenario["csi_gamma"]))
            csi_result = simulate(csi_tech, weather, base_cfg, npts=npts)
            perov_result = baseline_perov
            gamma_model = _gamma_pmax(csi_tech)
        elif "config" in scenario:
            cfg = replace(base_cfg, **scenario["config"])
            csi_result = simulate(CSI_MODERN, weather, cfg, npts=npts)
            perov_result = simulate(PEROVSKITE, weather, cfg, npts=npts)
            gamma_model = baseline_gamma
        elif "csi_config" in scenario:
            csi_cfg = replace(base_cfg, **scenario["csi_config"])
            csi_result = simulate(CSI_MODERN, weather, csi_cfg, npts=npts)
            perov_result = baseline_perov
            gamma_model = baseline_gamma
        else:
            raise ValueError(f"Unknown scenario definition: {scenario}")
        rows.append(
            _result_row(
                anchor,
                weather,
                scenario,
                csi_result,
                perov_result,
                gamma_model,
            )
        )
    return rows


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def _shard_path(shard_dir: Path, adcode: str) -> Path:
    return shard_dir / f"{adcode}.csv"


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def _valid_shard(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        frame = pd.read_csv(path, encoding="utf-8-sig", dtype={"adcode": str})
    except Exception:
        return False
    return (
        len(frame) == len(SCENARIOS)
        and frame["scenario"].tolist() == SCENARIO_ORDER
    )


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def _write_shard(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8-sig")


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def _consolidate(anchors: list[CityAnchor], shard_dir: Path) -> pd.DataFrame:
    frames = []
    missing = []
    for anchor in anchors:
        path = _shard_path(shard_dir, anchor.adcode)
        if _valid_shard(path):
            frames.append(pd.read_csv(
                path, encoding="utf-8-sig", dtype={"adcode": str},
            ))
        else:
            missing.append(anchor.adcode)
    if missing:
        raise RuntimeError(
            f"Sensitivity is incomplete; missing {len(missing)} anchors: "
            + ", ".join(missing[:8])
        )
    rows = pd.concat(frames, ignore_index=True)
    rows["scenario"] = pd.Categorical(
        rows["scenario"], categories=SCENARIO_ORDER, ordered=True,
    )
    return rows.sort_values(["scenario", "adcode"]).reset_index(drop=True)


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def build_rows(input_path: Path = OUT_CSV) -> pd.DataFrame:
    """Load the generated full-hourly sensitivity rows."""
    rows = pd.read_csv(
        input_path, encoding="utf-8-sig", dtype={"adcode": str},
    )
    rows["scenario"] = pd.Categorical(
        rows["scenario"], categories=SCENARIO_ORDER, ordered=True,
    )
    return rows.sort_values(["scenario", "adcode"]).reset_index(drop=True)


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def summarize(rows: pd.DataFrame) -> pd.DataFrame:
    """Summarise spatial magnitude, win count and inversion direction."""
    base = rows.loc[rows["scenario"] == "baseline", [
        "adcode", "ghi_kwh_m2",
    ]].copy()
    base["ghi_quartile"] = pd.qcut(
        base["ghi_kwh_m2"],
        4,
        labels=["Q1 low", "Q2", "Q3", "Q4 high"],
    )
    quartiles = base.set_index("adcode")["ghi_quartile"]
    summary_rows = []
    for scenario in SCENARIO_ORDER:
        group = rows[rows["scenario"] == scenario].copy()
        group["ghi_quartile"] = group["adcode"].map(quartiles)
        advantage = group["perovskite_advantage_pct"].to_numpy(float)
        ghi = group["ghi_kwh_m2"].to_numpy(float)
        q1 = float(group.loc[
            group["ghi_quartile"] == "Q1 low",
            "perovskite_advantage_pct",
        ].median())
        q4 = float(group.loc[
            group["ghi_quartile"] == "Q4 high",
            "perovskite_advantage_pct",
        ].median())
        meta = SCENARIO_LOOKUP[scenario]
        p10, p25, median, p75, p90 = np.percentile(
            advantage, [10, 25, 50, 75, 90],
        )
        corr = float(np.corrcoef(ghi, advantage)[0, 1])
        summary_rows.append({
            "scenario": scenario,
            "scenario_label": meta["label"],
            "scenario_group": meta["group"],
            "scenario_description": meta["description"],
            "city_count": int(len(group)),
            "perovskite_win_count": int(np.sum(advantage > 0.0)),
            "perovskite_win_share": float(np.mean(advantage > 0.0)),
            "median_advantage_pct": float(median),
            "p10_advantage_pct": float(p10),
            "p25_advantage_pct": float(p25),
            "p75_advantage_pct": float(p75),
            "p90_advantage_pct": float(p90),
            "min_advantage_pct": float(np.min(advantage)),
            "max_advantage_pct": float(np.max(advantage)),
            "irradiance_advantage_correlation": corr,
            "low_ghi_q1_median_pct": q1,
            "high_ghi_q4_median_pct": q4,
            "low_minus_high_ghi_pct_points": q1 - q4,
            "direction_preserved": bool(corr < 0.0 and q1 > q4),
        })
    return pd.DataFrame(summary_rows)


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def _plot(summary: pd.DataFrame, rows: pd.DataFrame) -> None:
    viz.setup_en(font_size=8)
    plt.rcParams.update({
        "axes.titlesize": 9.5,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7.2,
        "legend.fontsize": 6.8,
    })
    colors = {
        "baseline": "#202020",
        "device": "#6f6f6f",
        "mounting": "#d6604d",
        "inverter": "#2878b5",
        "boundary": "#2a9d4a",
    }
    summary = summary.set_index("scenario").loc[SCENARIO_ORDER].reset_index()
    y = np.arange(len(summary))
    fig, axes = plt.subplots(
        1, 2, figsize=(7.25, 4.25), gridspec_kw={"width_ratios": [1.16, 1.0]},
    )

    ax = axes[0]
    ax.axvline(0.0, color="#333333", lw=0.8, zorder=0)
    for index, row in summary.iterrows():
        color = colors[row["scenario_group"]]
        ax.plot(
            [row["p10_advantage_pct"], row["p90_advantage_pct"]],
            [index, index],
            color=color,
            alpha=0.36,
            lw=1.6,
            solid_capstyle="round",
        )
        ax.plot(
            [row["p25_advantage_pct"], row["p75_advantage_pct"]],
            [index, index],
            color=color,
            lw=4.2,
            solid_capstyle="round",
        )
        ax.scatter(
            row["median_advantage_pct"],
            index,
            s=28,
            color=color,
            edgecolor="white",
            linewidth=0.45,
            zorder=3,
        )
        ax.annotate(
            f"{int(row['perovskite_win_count'])}/337",
            (row["p90_advantage_pct"], index),
            xytext=(4, 0),
            textcoords="offset points",
            va="center",
            fontsize=6.5,
            color="#444444",
        )
    ax.set_yticks(y)
    ax.set_yticklabels(summary["scenario_label"])
    ax.invert_yaxis()
    ax.set_xlabel("Perovskite advantage over modern c-Si (%)")
    ax.set_title("(a) Full-hourly sensitivity", loc="left", fontweight="bold")
    ax.grid(axis="x", alpha=0.22, lw=0.4)
    ax.grid(axis="y", visible=False)
    ax.text(
        0.98,
        0.015,
        "dot: median   thick: IQR   thin: P10–P90\nright labels: perovskite wins",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=6.3,
        color="#666666",
    )

    ax = axes[1]
    ax.axvline(0.0, color="#333333", lw=0.8, zorder=0)
    for index, row in summary.iterrows():
        color = colors[row["scenario_group"]]
        q4 = row["high_ghi_q4_median_pct"]
        q1 = row["low_ghi_q1_median_pct"]
        ax.plot([q4, q1], [index, index], color=color, lw=1.4, alpha=0.65)
        ax.scatter(
            q4,
            index,
            marker="^",
            s=28,
            facecolor="white",
            edgecolor=color,
            linewidth=0.9,
            zorder=3,
        )
        ax.scatter(q1, index, marker="o", s=24, color=color, zorder=3)
        ax.annotate(
            f"r={row['irradiance_advantage_correlation']:.2f}",
            (max(q1, q4), index),
            xytext=(4, 0),
            textcoords="offset points",
            va="center",
            fontsize=6.5,
            color="#444444",
        )
    ax.set_yticks(y)
    ax.set_yticklabels([])
    ax.invert_yaxis()
    ax.set_xlabel("Quartile median advantage (%)")
    ax.set_title("(b) Low-GHI inversion", loc="left", fontweight="bold")
    ax.grid(axis="x", alpha=0.22, lw=0.4)
    ax.grid(axis="y", visible=False)
    ax.scatter([], [], marker="o", s=24, color="#555555", label="Q1 low GHI")
    ax.scatter(
        [], [], marker="^", s=28, facecolor="white", edgecolor="#555555",
        label="Q4 high GHI",
    )
    ax.legend(loc="lower right", frameon=False)

    fig.suptitle(
        "Modern c-Si comparator and system sensitivity (337 city anchors)",
        fontsize=10.2,
        fontweight="bold",
        x=0.06,
        ha="left",
        y=1.01,
    )
    fig.subplots_adjust(left=0.19, right=0.98, bottom=0.13, top=0.90, wspace=0.18)
    OUT_FIGURE.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUT_FIGURE.with_suffix(".png"), dpi=300, bbox_inches="tight")
    fig.savefig(OUT_FIGURE.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def run(
    *,
    catalog: Path = DEFAULT_CITY_CATALOG,
    workers: int = min(4, os.cpu_count() or 1),
    npts: int = 40,
    cache_dir: str = "data/tmy_cache",
    shard_dir: Path = DEFAULT_SHARD_DIR,
    allow_download: bool = True,
    refresh: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    anchors = load_city_catalog(catalog)
    if refresh:
        for anchor in anchors:
            path = _shard_path(shard_dir, anchor.adcode)
            if path.exists():
                path.unlink()
    pending = [
        anchor for anchor in anchors
        if not _valid_shard(_shard_path(shard_dir, anchor.adcode))
    ]
    print(
        f"Modern-c-Si sensitivity: {len(anchors)} anchors, "
        f"{len(pending)} pending, workers={workers}, npts={npts}",
        flush=True,
    )
    failures = []
    pool_type = ThreadPoolExecutor if 'ipykernel' in sys.modules else ProcessPoolExecutor
    with pool_type(max_workers=max(1, workers)) as executor:
        futures = {
            executor.submit(
                _simulate_anchor,
                anchor,
                max(20, npts),
                cache_dir,
                allow_download,
            ): anchor
            for anchor in pending
        }
        for count, future in enumerate(as_completed(futures), start=1):
            anchor = futures[future]
            try:
                rows = future.result()
                _write_shard(_shard_path(shard_dir, anchor.adcode), rows)
                print(
                    f"[{count}/{len(pending)}] {anchor.province} "
                    f"{anchor.city} complete",
                    flush=True,
                )
            except Exception as exc:
                failures.append({
                    "adcode": anchor.adcode,
                    "city": anchor.city,
                    "province": anchor.province,
                    "error": str(exc),
                    "traceback": traceback.format_exc(limit=8),
                })
                print(
                    f"FAILED {anchor.adcode} {anchor.city}: {exc}",
                    file=sys.stderr,
                    flush=True,
                )
    if failures:
        failure_path = ROOT / "outputs" / "si_silicon_system_sensitivity_failures.csv"
        pd.DataFrame(failures).to_csv(
            failure_path, index=False, encoding="utf-8-sig",
        )
        raise RuntimeError(
            f"{len(failures)} sensitivity anchors failed; see {failure_path}"
        )
    rows = _consolidate(anchors, shard_dir)
    summary = summarize(rows)
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    rows.to_csv(
        OUT_CSV,
        index=False,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_MINIMAL,
    )
    summary.to_csv(
        OUT_SUMMARY,
        index=False,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_MINIMAL,
    )
    _plot(summary, rows)
    return rows, summary


In [ ]:
%%plot_module scripts.silicon_baseline_sensitivity
def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--catalog", type=Path, default=DEFAULT_CITY_CATALOG)
    parser.add_argument("--workers", type=int, default=min(4, os.cpu_count() or 1))
    parser.add_argument("--npts", type=int, default=40)
    parser.add_argument("--cache-dir", default="data/tmy_cache")
    parser.add_argument("--shard-dir", type=Path, default=DEFAULT_SHARD_DIR)
    parser.add_argument("--offline", action="store_true")
    parser.add_argument("--refresh", action="store_true")
    args = parser.parse_args()
    rows, summary = run(
        catalog=args.catalog,
        workers=args.workers,
        npts=args.npts,
        cache_dir=args.cache_dir,
        shard_dir=args.shard_dir,
        allow_download=not args.offline,
        refresh=args.refresh,
    )
    print(f"Wrote {len(rows)} rows to {OUT_CSV}")
    print(summary[[
        "scenario_label",
        "median_advantage_pct",
        "perovskite_win_count",
        "irradiance_advantage_correlation",
        "direction_preserved",
    ]].to_string(index=False))
    return 0


In [ ]:
run_figure_module('scripts.silicon_baseline_sensitivity', RUNTIME)
